# J-lens / Antidoom baseline — `LiquidAI/LFM2-2.6B`

**Not** the blog's private early LFM2.5-2.6B. **Not** Antidoom-trained.

| Setting | Value |
|---|---|
| Model | `LiquidAI/LFM2-2.6B` |
| Thinking | leave **default** (dynamic hybrid reasoning — do not force-disable) |
| Prompts | 200 stratified antidoom-mix reasoning sources, seed=42 |
| max_new_tokens | **4000** |
| temperature | 0.01 |
| Backend | **vLLM** preferred (`fp8` → fallback `bfloat16`) |

Connect this notebook to a **Google Colab GPU kernel** (Colab extension), or Runtime→GPU in browser Colab.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Mithilyaganti/jlens-doom-loop-analysis.git"
PROJECT = Path("/content/j-lens")

# Always sync latest code from GitHub (fixes stale clone missing jlens vendor fix)
if (PROJECT / ".git").is_dir():
    r = subprocess.run(
        ["git", "-C", str(PROJECT), "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
    print(r.stdout or r.stderr)
    if r.returncode != 0:
        shutil.rmtree(PROJECT, ignore_errors=True)

if not (PROJECT / ".git").is_dir():
    print("Cloning", REPO_URL)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)],
        check=True,
    )

os.chdir(PROJECT)
marker = PROJECT / "jspace" / "vendor_bootstrap.py"
if not marker.is_file():
    raise RuntimeError(f"Repo outdated — missing {marker}. Re-run this cell.")

rev = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    cwd=PROJECT,
).stdout.strip()
print("PROJECT", PROJECT.resolve())
print("git", rev)
print("prompt sample", (PROJECT / "results" / "prompt_sample_ids.json").is_file())

In [ ]:
import os
import sys

os.chdir("/content/j-lens")
sys.path.insert(0, "/content/j-lens")

%pip install -q -U "pandas>=2.1,<2.4" transformers accelerate bitsandbytes datasets huggingface_hub \
    scipy statsmodels tqdm pyyaml safetensors sentencepiece matplotlib seaborn

try:
    %pip install -q vllm
    print("vllm ok")
except Exception as e:
    print("vllm install failed — will use HF:", e)

# Required: open-jlens vendor (not in git clone)
!python /content/j-lens/scripts/00_colab_vendors.py

# Verify before baseline — fail fast if jlens still missing
from jspace.vendor_bootstrap import ensure_jlens_importable
ensure_jlens_importable()
import jlens
print("jlens OK", jlens.__file__)

import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
import os
from pathlib import Path

os.chdir("/content/j-lens")

os.environ["JLENS_MODEL"] = "LiquidAI/LFM2-2.6B"
os.environ["JLENS_BASELINE_PROMPTS"] = "200"
os.environ["JLENS_SAMPLE_SEED"] = "42"
os.environ["JLENS_MAX_NEW_TOKENS"] = "4000"
os.environ["JLENS_TEMPERATURE"] = "0.01"
os.environ["JLENS_BACKEND"] = "vllm"
os.environ["JLENS_VLLM_DTYPE"] = "fp8"
os.environ["JLENS_MAX_MODEL_LEN"] = "6000"
os.environ["JLENS_HF_QUANTIZE"] = "0"
os.environ["PYTHONPATH"] = "/content/j-lens"

print({k: os.environ[k] for k in os.environ if k.startswith("JLENS_")})

In [ ]:
import os
os.chdir("/content/j-lens")
os.environ["PYTHONPATH"] = "/content/j-lens"

# 200-prompt baseline (resumes from checkpoint if interrupted)
!python /content/j-lens/scripts/run_lfm2_26b.py

In [ ]:
import json
from pathlib import Path

summary = Path("results/baseline_pass_summary.json")
report = Path("results/RUN_REPORT_lfm2-2.6b_antidoom_mix_200.md")
ckpt = Path("results/checkpoints/baseline_pass_lfm2-2.6b.json")

if ckpt.is_file():
    n = len(json.loads(ckpt.read_text())["completed_prompt_ids"])
    print(f"checkpoint progress: {n}/200")
if summary.is_file():
    s = json.loads(summary.read_text())
    print(f"loops: {s.get('n_loop')} rate={s.get('loop_rate', 0):.1%} backend={s.get('backend')}")
if report.is_file():
    print("\n--- REPORT ---\n")
    print(report.read_text())
else:
    !python scripts/write_lfm_report.py
    if report.is_file():
        print(report.read_text())

## After baseline: J-lens fit → Exp1–3

Only after a real lens is saved to `lenses/lfm2-2.6b.pt`:

```bash
python scripts/01_workspace_band.py
python scripts/03_exp1_static_geometry.py
python scripts/04_exp2_dynamic.py
python scripts/05_exp3_causal.py
```

Update those scripts' hard-coded `qwen3.5-4b` artifact names to use `jspace.model_config.artifact_paths()` if not already done.